In [1]:
import geopandas as gpd
import pandas as pd

# Paths
LOD2_NEW = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings.gpkg"
NHDA_PATH = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\NHDA_residential_wsf2015_max10pct.gpkg"
VG250_PATH = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
VG250_LAYER = "vg250_krs"

# Load data
print("Loading new buildings ...")
gdf_new = gpd.read_file(LOD2_NEW)
print(f"  {len(gdf_new):,} buildings | CRS: {gdf_new.crs}")

print("Loading NHDA polygons ...")
gdf_nhda = gpd.read_file(NHDA_PATH).to_crs(gdf_new.crs)
print(f"  {len(gdf_nhda):,} NHDA polygons")

print("Loading VG250 districts ...")
lkr_raw = gpd.read_file(VG250_PATH, layer=VG250_LAYER).to_crs(gdf_new.crs)
lkr_by = lkr_raw[lkr_raw["AGS"].astype(str).str.startswith("09")].copy()

# Detect a type/label column (often contains Landkreis / Kreisfreie Stadt)
type_candidates = [c for c in lkr_by.columns if "bez" in c.lower() or "art" in c.lower()]
type_col = type_candidates[0] if type_candidates else None

if type_col:
    lk_type_series = lkr_by[type_col].astype(str)
else:
    # Fallback if no explicit type column exists
    lk_type_series = lkr_by["GEN"].astype(str)

lkr_by["lk_type"] = lk_type_series.apply(
    lambda x: "Landkreis" if ("landkreis" in x.lower() or "lkr" in x.lower()) else "Kreisfreie Stadt"
)

lkr = lkr_by[["AGS", "GEN", "lk_type", "geometry"]].rename(
    columns={"AGS": "lk_ags", "GEN": "lk_name"}
).copy()

print(f"  {len(lkr):,} districts (Bavaria)")
print("  lk_type distribution:")
print(lkr["lk_type"].value_counts())

# Filter residential with known subclass
gdf_res = gdf_new[gdf_new["res_subclass"].notna()].copy()
print(f"\nResidential with res_subclass: {len(gdf_res):,}")

# Assign district by centroid
gdf_res_c = gdf_res.copy()
gdf_res_c["geometry"] = gdf_res_c.geometry.centroid

joined_lkr = gpd.sjoin(
    gdf_res_c[["geometry"]],
    lkr[["lk_ags", "lk_name", "lk_type", "geometry"]],
    how="left",
    predicate="within",
)

missing = joined_lkr["lk_ags"].isna()
if missing.any():
    nearest = gpd.sjoin_nearest(
        gdf_res_c.loc[missing, ["geometry"]],
        lkr[["lk_ags", "lk_name", "lk_type", "geometry"]],
        how="left",
    )
    joined_lkr.loc[missing, "lk_ags"] = nearest["lk_ags"].values
    joined_lkr.loc[missing, "lk_name"] = nearest["lk_name"].values
    joined_lkr.loc[missing, "lk_type"] = nearest["lk_type"].values

gdf_res["lk_ags"] = joined_lkr["lk_ags"].values
gdf_res["lk_name"] = joined_lkr["lk_name"].values
gdf_res["lk_type"] = joined_lkr["lk_type"].values

# Inside NHDA?
nhda_union = gdf_nhda[["geometry"]].copy()
nhda_union["_nhda"] = 1

in_nhda = gpd.sjoin(
    gdf_res[["geometry", "lk_ags", "lk_name"]],
    nhda_union,
    how="left",
    predicate="within",
)
gdf_res["in_nhda"] = in_nhda["_nhda"].notna().values

# Aggregate per district
agg = (
    gdf_res.groupby(["lk_ags", "lk_name", "lk_type"])["in_nhda"]
    .agg(
        n_in_nhda=lambda x: x.sum(),
        n_outside_nhda=lambda x: (~x).sum(),
        n_total=lambda x: len(x),
    )
    .reset_index()
)

agg["ratio_in_nhda"] = (agg["n_in_nhda"] / agg["n_total"]).round(4)
agg["ratio_outside_nhda"] = (agg["n_outside_nhda"] / agg["n_total"]).round(4)
agg = agg.sort_values("lk_ags").reset_index(drop=True)

print("\nResult (first 10 rows):")
print(agg.head(10).to_string(index=False))
print(f"\nTotal: {agg['n_total'].sum():,} residential buildings across {len(agg)} districts")

agg

Loading new buildings ...
  1,811,517 buildings | CRS: EPSG:25832
Loading NHDA polygons ...
  839 NHDA polygons
Loading VG250 districts ...
  96 districts (Bavaria)
  lk_type distribution:
lk_type
Landkreis           71
Kreisfreie Stadt    25
Name: count, dtype: int64

Residential with res_subclass: 227,336

Result (first 10 rows):
lk_ags                 lk_name          lk_type  n_in_nhda  n_outside_nhda  n_total  ratio_in_nhda  ratio_outside_nhda
 09161              Ingolstadt Kreisfreie Stadt        324            1748     2072         0.1564              0.8436
 09162                 München Kreisfreie Stadt        162            6981     7143         0.0227              0.9773
 09163               Rosenheim Kreisfreie Stadt          0             489      489         0.0000              1.0000
 09171               Altötting        Landkreis        268            2015     2283         0.1174              0.8826
 09172    Berchtesgadener Land        Landkreis         70             

,lk_ags,lk_name,lk_type,n_in_nhda,n_outside_nhda,n_total,ratio_in_nhda,ratio_outside_nhda
0,09161,Ingolstadt,Kreisfreie Stadt,324,1748,2072,0.1564,0.8436
1,09162,München,Kreisfreie Stadt,162,6981,7143,0.0227,0.9773
2,09163,Rosenheim,Kreisfreie Stadt,0,489,489,0.0000,1.0000
3,09171,Altötting,Landkreis,268,2015,2283,0.1174,0.8826
4,09172,Berchtesgadener Land,Landkreis,70,982,1052,0.0665,0.9335
...,...,...,...,...,...,...,...,...
91,09776,Lindau (Bodensee),Landkreis,79,1177,1256,0.0629,0.9371
92,09777,Ostallgäu,Landkreis,361,3240,3601,0.1002,0.8998
93,09778,Unterallgäu,Landkreis,434,3838,4272,0.1016,0.8984
94,09779,Donau-Ries,Landkreis,560,3011,3571,0.1568,0.8432


In [2]:
agg.to_csv(r"C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\nhda_vs_infill_per_lkr.csv", index=False, encoding='utf-8-sig')

In [3]:

# Save with English column names for LaTeX
agg_en = agg.copy()

# Debug: Check lk_type values
print("Debug: lk_type values in agg:")
print(agg[["lk_ags", "lk_name", "lk_type"]].head(10))

# Add (Lkr.) to district names if it's a Landkreis
agg_en["District Name"] = agg_en.apply(
    lambda row: f"{row['lk_name']} (Lkr.)" if row["lk_type"] == "Landkreis" else row["lk_name"],
    axis=1
)

print("\nDebug: After adding (Lkr.):")
print(agg_en[["lk_type", "District Name"]].head(10))

agg_en = agg_en.rename(columns={
    "lk_ags": "AGS",
    "n_in_nhda": "Inside NHDA",
    "n_outside_nhda": "Outside NHDA",
    "n_total": "Total Buildings",
    "ratio_in_nhda": "Ratio Inside",
    "ratio_outside_nhda": "Ratio Outside"
})

# Select only the columns we want for LaTeX
agg_en = agg_en[["AGS", "District Name", "Inside NHDA", "Outside NHDA", "Total Buildings", "Ratio Inside", "Ratio Outside"]]

agg_en.to_csv(
    r"C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\nhda_vs_infill_per_lkr_en.csv",
    index=False,
    encoding='utf-8-sig'
)
print("✓ Saved: nhda_vs_infill_per_lkr_en.csv")
print("\nFirst few rows:")
print(agg_en.head(10))


Debug: lk_type values in agg:
  lk_ags                  lk_name           lk_type
0  09161               Ingolstadt  Kreisfreie Stadt
1  09162                  München  Kreisfreie Stadt
2  09163                Rosenheim  Kreisfreie Stadt
3  09171                Altötting         Landkreis
4  09172     Berchtesgadener Land         Landkreis
5  09173  Bad Tölz-Wolfratshausen         Landkreis
6  09174                   Dachau         Landkreis
7  09175                Ebersberg         Landkreis
8  09176                Eichstätt         Landkreis
9  09177                   Erding         Landkreis

Debug: After adding (Lkr.):
            lk_type                   District Name
0  Kreisfreie Stadt                      Ingolstadt
1  Kreisfreie Stadt                         München
2  Kreisfreie Stadt                       Rosenheim
3         Landkreis                Altötting (Lkr.)
4         Landkreis     Berchtesgadener Land (Lkr.)
5         Landkreis  Bad Tölz-Wolfratshausen (Lkr.)
6    

In [4]:
print(f"Mean ratio_in_nhda (per LKR):  {agg['ratio_in_nhda'].mean():.4f}")
print(f"Mean ratio_outside_nhda:       {agg['ratio_outside_nhda'].mean():.4f}")
print()
print("Weighted by total buildings:")
weighted = (agg["ratio_in_nhda"] * agg["n_total"]).sum() / agg["n_total"].sum()
print(f"  Weighted mean ratio_in_nhda: {weighted:.4f}")


Mean ratio_in_nhda (per LKR):  0.0805
Mean ratio_outside_nhda:       0.9195

Weighted by total buildings:
  Weighted mean ratio_in_nhda: 0.0882
